In [ ]:
pip install requests beautifulsoup4 langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=c590aa18dddd0d6180a8bc575833c3e37ec0766de182179807dd631397c6f1eb
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
"""
LinkedIn Entry-Level Power/Electrical Engineer Job Scraper - Netherlands
Filters only ENGLISH job postings by reading the actual job description.
"""

import requests
from bs4 import BeautifulSoup
from langdetect import detect, LangDetectException
import json
import time
from dataclasses import dataclass, asdict


@dataclass
class Job:
    title: str
    company: str
    location: str
    posted: str
    job_url: str
    job_id: str
    language: str = "unknown"
    description_snippet: str = ""


def build_search_url(keyword: str, location: str, start: int = 0) -> str:
    """LinkedIn guest job search URL with entry-level filter."""
    base = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    params = {
        "keywords": keyword,
        "location": location,
        "f_E": "2",   # Entry level
        "start": start,
    }
    param_str = "&".join(f"{k}={requests.utils.quote(str(v))}" for k, v in params.items())
    return f"{base}?{param_str}"


def fetch_job_listings(url: str, headers: dict) -> list[Job]:
    """Fetch a page of job cards from LinkedIn search results."""
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  [!] Failed to fetch listings: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    cards = soup.find_all("div", class_="base-card")
    jobs = []

    for card in cards:
        try:
            title    = card.find("h3", class_="base-search-card__title")
            company  = card.find("h4", class_="base-search-card__subtitle")
            location = card.find("span", class_="job-search-card__location")
            posted   = card.find("time")
            link     = card.find("a", class_="base-card__full-link")
            job_id   = card.get("data-entity-urn", "").split(":")[-1]

            jobs.append(Job(
                title    = title.get_text(strip=True)   if title    else "N/A",
                company  = company.get_text(strip=True) if company  else "N/A",
                location = location.get_text(strip=True)if location else "N/A",
                posted   = posted.get_text(strip=True)  if posted   else "N/A",
                job_url  = link["href"].strip()          if link     else "N/A",
                job_id   = job_id,
            ))
        except Exception as e:
            print(f"  [!] Skipped card: {e}")

    return jobs


def fetch_job_description(job_url: str, headers: dict) -> str:
    """
    Fetch the full job description text from the individual job page.
    Returns empty string on failure.
    """
    try:
        resp = requests.get(job_url, headers=headers, timeout=12)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Primary target: the description div
        desc_div = soup.find("div", class_="description__text")
        if desc_div:
            return desc_div.get_text(separator=" ", strip=True)

        # Fallback: grab all paragraph text
        paragraphs = soup.find_all(["p", "li"])
        return " ".join(p.get_text(strip=True) for p in paragraphs)

    except requests.RequestException as e:
        print(f"    [!] Could not fetch job page: {e}")
        return ""


def detect_language(text: str) -> str:
    """Detect language of a text string. Returns ISO code or 'unknown'."""
    if not text or len(text.strip()) < 30:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


def scrape_english_ee_jobs(
    keywords: list[str],
    location: str = "Netherlands",
    max_pages: int = 3,
    request_delay: float = 2.5,
    desc_delay: float = 1.5,
    save_to_json: bool = True,
) -> list[Job]:
    """
    Main scraper. For each job found:
      1. Fetches the full description page
      2. Detects its language
      3. Keeps only English postings
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    }

    all_jobs: list[Job] = []
    seen_ids: set[str] = set()

    for keyword in keywords:
        print(f"\n🔍 Keyword: '{keyword}'")
        for page in range(max_pages):
            start = page * 25
            url = build_search_url(keyword, location, start)
            print(f"  Page {page + 1} (offset {start})... ", end="", flush=True)

            listings = fetch_job_listings(url, headers)
            if not listings:
                print("no results or end of pages.")
                break
            print(f"{len(listings)} listings found.")

            for job in listings:
                if not job.job_id or job.job_id in seen_ids:
                    continue
                seen_ids.add(job.job_id)

                # Fetch and check description language
                print(f"    Checking: {job.title[:55]:<55} ", end="", flush=True)
                desc = fetch_job_description(job.job_url, headers)
                lang = detect_language(desc)
                job.language = lang
                job.description_snippet = desc[:300]  # Store a preview

                if lang == "en":
                    all_jobs.append(job)
                    print("✅ English")
                else:
                    print(f"❌ Skipped ({lang})")

                time.sleep(desc_delay)

            time.sleep(request_delay)

    print(f"\n✅ Total English-language jobs found: {len(all_jobs)}")

    if save_to_json and all_jobs:
        with open("english_ee_jobs.json", "w", encoding="utf-8") as f:
            json.dump([asdict(j) for j in all_jobs], f, indent=2, ensure_ascii=False)
        print("💾 Results saved to english_ee_jobs.json")

    return all_jobs


def print_results(jobs: list[Job]) -> None:
    """Pretty-print job results to terminal."""
    print("\n" + "=" * 72)
    print(f"{'ENGLISH POWER/ELECTRICAL EE JOBS — NETHERLANDS':^72}")
    print("=" * 72)
    for i, job in enumerate(jobs, 1):
        print(f"\n[{i}] {job.title}")
        print(f"    🏢 {job.company}")
        print(f"    📍 {job.location}")
        print(f"    🕒 {job.posted}")
        print(f"    🔗 {job.job_url}")
        if job.description_snippet:
            snippet = job.description_snippet[:200].replace("\n", " ")
            print(f"    📄 {snippet}...")
    print("\n" + "=" * 72)


if __name__ == "__main__":
    # Power engineering + related electrical disciplines (all in English)
    SEARCH_KEYWORDS = [
        # Core power engineering
        "junior power engineer",
        "entry level power engineer",
        "graduate power engineer",
        # Electrical engineering
        "junior electrical engineer",
        "graduate electrical engineer",
        # Power systems & grid
        "power systems engineer",
        "grid engineer junior",
        "energy systems engineer junior",
        # High voltage / protection
        "high voltage engineer junior",
        "protection relay engineer junior",
        # Renewables (big in NL)
        "junior wind energy engineer",
        "junior solar energy engineer",
        "offshore wind engineer junior",
        # Electronics & hardware
        "junior hardware engineer",
        "junior electronics engineer",
        # Embedded / firmware (EE adjacent)
        "junior embedded engineer",
        "junior firmware engineer",
        # Power electronics
        "power electronics engineer junior",
        "junior drives engineer",
        # Instrumentation & control
        "junior instrumentation engineer",
        "junior control systems engineer",
    ]

    jobs = scrape_english_ee_jobs(
        keywords=SEARCH_KEYWORDS,
        location="Netherlands",
        max_pages=2,       # 2 pages × 25 = up to 50 results per keyword
        request_delay=2.5, # Delay between search pages
        desc_delay=1.5,    # Delay between individual job page fetches
        save_to_json=True,
    )

    print_results(jobs)


🔍 Keyword: 'junior power engineer'
  Page 1 (offset 0)... 10 listings found.
    Checking: PV Field Engineer                                       ✅ English
    Checking: (Junior) Project Engineer                               ✅ English
    Checking: ELECTRICAL DESIGN ENGINEER                              ✅ English
    Checking: Electrical Engineer                                     ✅ English
    Checking: Field Application Engineer-Power Supplies (on-site)     ✅ English
    Checking: Junior FPGA Design Engineer                             ✅ English
    Checking: Electrical Engineer                                     ❌ Skipped (nl)
    Checking: Industrial Automation, Electrical, Electronic Engineer  ✅ English
    Checking: 电气工程师——西安公司                                             ❌ Skipped (zh-cn)
    Checking: Junior Project Engineer                                 ❌ Skipped (nl)
  Page 2 (offset 25)... 10 listings found.
    Checking: E-PLAN ENGINEER                                